# exp072 — exp045 + tarea auxiliar fastMRI+ de localización de lesiones en `split_v1_seed42` fold 0

Padre: exp045_raptor_aug_espacial. Única variable: la pérdida de entrenamiento pasa de `L_main` a
`L_main + 0.3 · L_aux`, donde `L_aux` localiza cajas fastMRI+ de 11 clases mediante una cabeza densa separada sobre
el mismo backbone CoAtNet. Por paso se usan 48 ventanas coronales 2.5D (4 volúmenes × 12 ventanas). La cabeza auxiliar
no forma parte del SWA ni de los checkpoints usados para inferencia. Todo lo demás permanece idéntico a exp045.

SPEC: `SPEC.md` en esta carpeta. La AUC de presencia en el holdout fastMRI+ es solo diagnóstica y nunca selecciona
época, checkpoint ni hiperparámetro. Gold conserva su papel exclusivo de veto.

Celdas fijas: **0** parámetros · **1** setup · **2** datos y split · **3** modelo · **4** entrenamiento · **5** evaluación ·
**6** guardar · **7** verdict.


## 0. Parámetros (única celda con valores)

In [ ]:
EXP = "exp072_raptor_fm_aux"
PADRE = "exp045"
FOLDS = [0]
SEED = 42
SPLIT = "split_v1_seed42"
SELECT_ON = "pseudo"
SMOKE = True
ALL_DATA = False          # True: entrena con los 4349 no-gold, sin validación; SWA sobre SWA_EPOCHS fijas
SWA_EPOCHS = []           # solo ALL_DATA: épocas (0-index) a promediar, fijadas con lo aprendido en fold 0

CORPUS = "/root/corpus96_384"     # disco local del pod (build_corpus96.py)
LABELS_SOFT = "/workspace/data/labels/public/dreaddevelopment/rsna-knee-labels/labels_llm_soft.parquet"  # fuente 1 (la del padre)
L3_CSV = "/workspace/data/labels/public/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv"  # exp031: fuente 3
L4_CSV = "/workspace/data/labels/public/lixin73/rsna-knee-llm-report-labels-sol56/labels_llm_gpt56sol.csv"  # exp031: fuente 4
L2_CSV = "/workspace/data/labels/public/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"  # vara y fuente 2
V1_CSV = "/workspace/data/labels/report_labels_v1.csv"
ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
IMG = 384                 # lado guardado en el corpus
RES = 320                 # entrada = recorte central RES del corte IMG (como la inferencia nartaa)
NORM = "imagenet"
K = 12
K_EVAL = 24
EPOCHS = 16
BATCH_SIZE = 8
BB_LR = 3e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.02
PCT_START = 0.15
GRAD_CLIP = 3.0
TOPK = 3                  # SWA = media de las TOPK mejores épocas por pseudo L2
L2_THRESHOLD = 0.5
GRAD_CKPT = True
AUG_FLIP_P = 0.5          # exp045: espejo L/R anatómico por estudio (padre: sin espejo)
AUG_ROT_DEG = 10.0        # exp045: afín por estudio, rotación U(-10°, 10°)
AUG_SCALE = [0.9, 1.1]    # exp045: escala U(0.9, 1.1)
AUG_TRANS = 0.06          # exp045: traslación U(-6 %, 6 %) del lado, en x e y
N_WORKERS = 12
LOG_EVERY = 50

# exp072: tarea auxiliar fastMRI+; con peso 0 no se carga el store ni se crea/ejecuta la cabeza auxiliar.
FM_DIR = "/workspace/data/external/fastmri_plus"
FM_LOCAL = "/root/fm384"
FM_AUX_WEIGHT = 0.3
FM_AUX_WINDOWS = 48
FM_VOLS_PER_STEP = 4
FM_BOXED_FRAC = 0.75
FM_POS_WEIGHT = 8.0
FM_PRESENCE_MIN = 0.25
FM_HOLDOUT_FRAC = 0.05
FM_WORKERS = 2
NOTES = "exp045 + tarea auxiliar fastMRI+ de cajas (48 ventanas/paso, 11 clases, lambda 0.3); selección pseudo L2"


## 1. Setup

In [ ]:
import copy
import json
import math
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))
from src.knee.config import LABELS, current_user
from src.knee.experiments import decision, git_info, guardar_metricas, leer_metricas, seed_everything
from src.knee.metrics import mean_auc
from src.knee.splits import load_split

assert SELECT_ON == "pseudo", "Gold nunca selecciona checkpoints (docs/06_SDD.md)"
assert (REPO / "experiments" / EXP / "SPEC.md").exists(), "Primero escribe la SPEC del experimento"
assert not ALL_DATA or len(FOLDS) == 1
AUTHOR = current_user()
RESULTS = REPO / "results" / EXP
RESULTS.mkdir(parents=True, exist_ok=True)
GIT = git_info(REPO)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
if SMOKE:
    EPOCHS = 1
TAG = "alldata" if ALL_DATA else "f"
seed_everything(SEED)
print(f"{EXP} · {AUTHOR} · git {GIT['commit']} ({GIT['branch']}) · "
      f"{torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu'} · timm {timm.__version__} · torch {torch.__version__}")
if GIT["src_dirty"]:
    print("OJO: src/ o SPLITS/ tienen cambios sin commit; el resultado no será reproducible.")

## 2. Datos y split (las revisiones detienen el notebook si algo está mal)

In [ ]:
def _read(path, cols=None):
    df = pd.read_parquet(path) if str(path).endswith(".parquet") else pd.read_csv(path)
    df["StudyInstanceUID"] = df.StudyInstanceUID.astype(str)
    return df.set_index("StudyInstanceUID")[cols or LABELS].astype(float)

split = load_split(SPLIT)
def _read_soft(path, name):
    df = pd.read_parquet(path) if str(path).endswith(".parquet") else pd.read_csv(path, low_memory=False)
    df["StudyInstanceUID"] = df.StudyInstanceUID.astype(str)
    assert not df.StudyInstanceUID.duplicated().any(), f"{name} con StudyInstanceUID duplicados"
    df = df.set_index("StudyInstanceUID")[LABELS].apply(pd.to_numeric, errors="coerce").astype(float)
    assert df.stack().between(0.0, 1.0).all(), f"{name} fuera de [0, 1]"
    return df

# exp031: etiquetas de ENTRENAMIENTO = promedio por celda de 4 fuentes blandas (si falta una fuente, media de las presentes)
SOURCES = {"raptor": _read_soft(LABELS_SOFT, "raptor"), "L2": _read_soft(L2_CSV, "L2"),
           "L3": _read_soft(L3_CSV, "L3"), "L4": _read_soft(L4_CSV, "L4")}
_all = sorted(set().union(*[s.index for s in SOURCES.values()]))
_stack = np.stack([s.reindex(_all)[LABELS].to_numpy(dtype=float) for s in SOURCES.values()])
_n_src = np.isfinite(_stack).sum(0)
soft = pd.DataFrame(np.where(_n_src > 0, np.nansum(_stack, 0) / np.maximum(_n_src, 1), np.nan),
                    index=pd.Index(_all, name="StudyInstanceUID"), columns=LABELS)
N_SRC = pd.DataFrame(_n_src, index=soft.index, columns=LABELS)
l2 = _read(L2_CSV)                    # vara interna: L2 binarizada > 0.5
v1 = _read(V1_CSV)                    # informativo
train_csv = pd.read_csv("/workspace/data/raw/train.csv")
train_csv["StudyInstanceUID"] = train_csv.StudyInstanceUID.astype(str)
gold = train_csv.dropna(subset=LABELS, how="all").set_index("StudyInstanceUID")[LABELS].astype(float)
gold_ids = list(gold.index)
assert len(gold) == 58 and set(gold_ids) == set(split.loc[split.is_gold == 1, "StudyInstanceUID"])
soft = soft.dropna(how="all")         # estudio sin ninguna fuente fuera; celda sin fuentes (NaN) se enmascara en la pérdida
assert np.nanmin(soft.values) >= 0 and np.nanmax(soft.values) <= 1
print(f"etiquetas de entrenamiento (consenso): {len(soft)} estudios · celdas NaN {int(soft.isna().sum().sum())}")

corpus_ids = np.load(f"{CORPUS}/ids.npy").astype(str)
corpus_masks = np.load(f"{CORPUS}/masks.npy")
ID2ROW = {u: i for i, u in enumerate(corpus_ids)}
D = corpus_masks.shape[1]
assert D == 96, D
meta = json.load(open(f"{CORPUS}/build_meta.json"))
print(f"corpus {len(corpus_ids)} × {D} · cortes válidos medios {corpus_masks.sum(1).mean():.1f} · "
      f"estudios vacíos {(corpus_masks.sum(1) == 0).sum()} · errores build {len(meta['errors'])}")
assert set(gold_ids) <= set(ID2ROW)

pseudo_bin = (l2 > L2_THRESHOLD).astype(float).where(l2.notna())
v1_bin = v1.where(v1.isin([0.0, 1.0]))

def fold_ids(fold):
    """train / val para un fold de split_v1; train = pool sin el fold ∩ etiquetas de entrenamiento (consenso)."""
    if ALL_DATA:
        tr = [u for u in soft.index if u not in set(gold_ids) and u in ID2ROW]
        return tr, []
    pool = split[split.fold >= 0]
    tr = [u for u in pool.loc[pool.fold != fold, "StudyInstanceUID"] if u in soft.index and u in ID2ROW]
    va = [u for u in pool.loc[pool.fold == fold, "StudyInstanceUID"] if u in ID2ROW]
    return tr, va

for f in FOLDS:
    tr, va = fold_ids(f)
    assert not set(tr) & set(gold_ids), "gold dentro del entrenamiento"
    assert not set(tr) & set(va)
    print(f"fold {f}: train {len(tr)} · val {len(va)} (L2 cubre {pseudo_bin.index.isin(va).sum()}) · gold {len(gold_ids)}")
    _ns = N_SRC.reindex(tr).fillna(0).to_numpy()
    print(f"  celdas de train por nº de fuentes (0..4): {[int((_ns == k).sum()) for k in range(5)]} · "
          f"cobertura por fuente (estudios de train): {({n: int(pd.Index(tr).isin(s.index).sum()) for n, s in SOURCES.items()})}")
    _tr = {n: s.reindex(tr)[LABELS].to_numpy() for n, s in SOURCES.items()}
    _agree = lambda a, b: np.mean([(x > 0.5) == (y > 0.5) for x, y in zip(_tr[a].ravel(), _tr[b].ravel())
                                   if np.isfinite(x) and np.isfinite(y)])
    print("  acuerdo binarizado (> 0.5) en train: " + " · ".join(f"{a}–{b} {_agree(a, b):.3f}" for a, b in
          [("L2", "L4"), ("L2", "L3"), ("L3", "L4"), ("raptor", "L2"), ("raptor", "L3"), ("raptor", "L4")]))
    print(f"  media del objetivo en train: consenso {np.nanmean(soft.loc[tr].to_numpy()):.3f} · " +
          " · ".join(f"{n} {np.nanmean(v):.3f}" for n, v in _tr.items()))


# exp072: store fastMRI+ y cajas. Se inicializa solo si la pérdida auxiliar está activa.
FM_CLASSES = ("meniscus_tear", "cart_partial", "cart_full", "bone_edema", "bone_fx_contusion",
              "effusion", "cyst", "acl_high", "acl_low", "collateral", "other")
FM_A = len(FM_CLASSES)


def fm_label_to_class(label):
    """Mapeo fastMRI+ -> orden de 11 clases de R7; espacios externos no cambian el resultado."""
    label = str(label).strip()
    lower = label.lower()
    if "artifact" in lower:
        return None
    exact = {
        "Meniscus Tear": "meniscus_tear", "Displaced Meniscal Tissue": "meniscus_tear",
        "Cartilage - Partial Thickness loss/defect": "cart_partial",
        "Cartilage - Full Thickness loss/defect": "cart_full",
        "Bone- Subchondral edema": "bone_edema",
        "Bone-Fracture/Contusion/dislocation": "bone_fx_contusion", "Joint Effusion": "effusion",
        "Periarticular cysts": "cyst", "Ligament - ACL High Grade Sprain": "acl_high",
        "Ligament - ACL Low Grade sprain": "acl_low",
    }
    if "MCL" in label.upper() or "LCL" in label.upper():
        return "collateral"
    return exact.get(label, "other")


FM_N_SLICES, FM_BOXES, FM_BOXED_SLICES = {}, {}, {}
FM_TRAIN_BOXED, FM_TRAIN_NEG, FM_HOLDOUT = [], [], []
FM_BOX_COUNTS = {name: 0 for name in FM_CLASSES}
if FM_AUX_WEIGHT > 0:
    import shutil

    fm_root, fm_local = Path(FM_DIR), Path(FM_LOCAL)
    for name in ("vol", "knee.csv", "knee_file_list.csv", "meta.jsonl"):
        assert (fm_root / name).exists(), f"fastMRI+ incompleto: falta {fm_root / name}"
    if not fm_local.is_dir():
        print(f"copiando fastMRI+ {fm_root / 'vol'} -> {fm_local} ...", flush=True)
        shutil.copytree(fm_root / "vol", fm_local)

    reviewed = sorted({line.strip().split(",")[0] for line in (fm_root / "knee_file_list.csv").read_text().splitlines()
                       if line.strip().startswith("file")})
    fm_meta = {}
    with open(fm_root / "meta.jsonl", encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                rec = json.loads(line)
                fm_meta[Path(str(rec["file"])).stem] = rec
    usable = sorted(name for name in reviewed if name in fm_meta and (fm_local / f"{name}.npy").exists())
    assert usable, "no hay volúmenes fastMRI+ utilizables"
    for name in usable:
        vol = np.load(fm_local / f"{name}.npy", mmap_mode="r")
        assert vol.dtype == np.uint8 and vol.ndim == 3 and tuple(vol.shape[1:]) == (IMG, IMG), (name, vol.shape, vol.dtype)
        assert vol.shape[0] == int(fm_meta[name]["n_slices"]) and vol.shape[0] >= 3, (name, vol.shape, fm_meta[name])
        FM_N_SLICES[name] = int(vol.shape[0])
        del vol

    ann = pd.read_csv(fm_root / "knee.csv")
    for col in ("slice", "x", "y", "width", "height"):
        ann[col] = pd.to_numeric(ann[col], errors="coerce")
    scale = IMG / 320.0
    usable_set = set(usable)
    for row in ann.loc[ann[["slice", "x", "y", "width", "height"]].notna().all(axis=1)].itertuples(index=False):
        name = Path(str(row.file).strip()).stem
        cls = fm_label_to_class(row.label)
        if name not in usable_set or cls is None:
            continue
        sl = int(row.slice)
        if not 0 <= sl < FM_N_SLICES[name]:
            continue
        x, y = float(row.x) * scale, float(row.y) * scale
        w, h = float(row.width) * scale, float(row.height) * scale
        a0 = max(w, 0.0) * max(h, 0.0)
        x0, y0, x1, y1 = np.clip([x, y, x + w, y + h], 0.0, float(IMG))
        if x1 <= x0 or y1 <= y0 or a0 <= 0:
            continue
        c = FM_CLASSES.index(cls)
        FM_BOXES.setdefault((name, sl), []).append((c, x0, y0, x1, y1, a0))
        FM_BOX_COUNTS[cls] += 1
    FM_BOXES = {key: np.asarray(value, np.float64) for key, value in FM_BOXES.items()}
    for name in usable:
        FM_BOXED_SLICES[name] = np.array(sorted(sl for (vol_name, sl) in FM_BOXES if vol_name == name), np.int64)

    boxed = sorted(name for name in usable if len(FM_BOXED_SLICES[name]))
    negative = sorted(set(usable) - set(boxed))
    rng_hold = np.random.default_rng(9800)
    def _fm_holdout_sample(pool):
        n = max(1, int(round(len(pool) * FM_HOLDOUT_FRAC)))
        return [str(x) for x in rng_hold.choice(np.asarray(pool), size=n, replace=False)]
    FM_HOLDOUT = sorted(_fm_holdout_sample(boxed) + _fm_holdout_sample(negative))
    hold_set = set(FM_HOLDOUT)
    FM_TRAIN_BOXED = [name for name in boxed if name not in hold_set]
    FM_TRAIN_NEG = [name for name in negative if name not in hold_set]
    assert FM_TRAIN_BOXED and FM_TRAIN_NEG and not hold_set & (set(FM_TRAIN_BOXED) | set(FM_TRAIN_NEG))
    print(pd.DataFrame({"boxes": FM_BOX_COUNTS}).rename_axis("fm_class"))
    print(f"fastMRI+: {len(usable)} volúmenes · train {len(FM_TRAIN_BOXED)} con cajas + {len(FM_TRAIN_NEG)} negativos · "
          f"holdout estratificado {len(FM_HOLDOUT)} · cajas {sum(FM_BOX_COUNTS.values())}")


## 3. Modelo (verbatim del script público: `build_backbone` + `RaptorClassifier`; dataset `StudyWindows`)

In [ ]:
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)
SLOT_SIZES = [26, 22, 18, 12, 18]     # exp045: slots del corpus (Sag fluid, Sag, Cor fluid, Cor, Ax), como el TTA nartaa
SAGITTAL_SLOTS = (0, 1)
SLOT_OF = np.repeat(np.arange(len(SLOT_SIZES)), SLOT_SIZES)
assert len(SLOT_OF) == D


class StudyWindows(Dataset):
    """Bolsa de ventanas 2.5D por estudio (3 cortes consecutivos -> RGB) desde el corpus memmap.
    Igual que el script público salvo: recorte central RES del corte IMG (en vez de resize) y la conversión a float,
    la ganancia ×U(0.9,1.1) y la normalización ImageNet se hacen en GPU (mismas operaciones)."""
    def __init__(self, ids, k, train):
        self.ids, self.k, self.train = list(ids), k, train
        self.vols = None

    def __len__(self):
        return len(self.ids)

    def _centers(self, valid, count):
        lo, hi = int(valid.min()), int(valid.max())
        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
        if not cs:
            cs = [max(1, min((lo + hi) // 2, D - 2))]
        if self.train:
            pool = cs * (count // len(cs) + 1)
            random.shuffle(pool)
            return pool[:count]
        idx = np.linspace(0, len(cs) - 1, count).round().astype(int)
        return [cs[i] for i in idx]

    def __getitem__(self, i):
        if self.vols is None:
            self.vols = np.load(f"{CORPUS}/vols.npy", mmap_mode="r")
        uid = self.ids[i]; row = ID2ROW[uid]
        valid = np.where(corpus_masks[row] > 0)[0]
        if len(valid) < 3:
            valid = np.arange(min(3, D))
        cs = np.array([max(1, min(c, D - 2)) for c in self._centers(valid, self.k)])
        lo = (IMG - RES) // 2
        vol = self.vols[row]
        tri = np.stack([vol[cs - 1, lo:lo + RES, lo:lo + RES], vol[cs, lo:lo + RES, lo:lo + RES],
                        vol[cs + 1, lo:lo + RES, lo:lo + RES]], 1)          # (K,3,RES,RES) uint8
        gain = 1.0 + (random.random() - 0.5) * 0.20 if self.train else 1.0
        sag = torch.from_numpy(np.isin(SLOT_OF[cs], SAGITTAL_SLOTS))     # exp045: eje del espejo por ventana (slot del corte central)
        return torch.from_numpy(np.ascontiguousarray(tri)), torch.tensor(gain, dtype=torch.float32), i, sag


def spatial_aug(x, sag):
    """exp045: espejo L/R anatómico (p AUG_FLIP_P) + afín (misma matriz para las K ventanas de un estudio), en GPU.
    Espejo = `_mirror_tri` del TTA nartaa: ventana con corte central sagital -> invertir los 3 cortes-canal; si no -> volteo W."""
    B, K_, C, H, W = x.shape
    flip = (torch.rand(B, device=x.device) < AUG_FLIP_P).view(B, 1, 1, 1, 1)
    sag = sag.to(x.device).view(B, K_, 1, 1, 1)
    x = torch.where(flip, torch.where(sag, x.flip(2), x.flip(-1)), x)
    ang = (torch.rand(B, device=x.device) * 2 - 1) * math.radians(AUG_ROT_DEG)
    scl = AUG_SCALE[0] + torch.rand(B, device=x.device) * (AUG_SCALE[1] - AUG_SCALE[0])
    tr = (torch.rand(B, 2, device=x.device) * 2 - 1) * AUG_TRANS * 2      # coords normalizadas [-1, 1]: lado = 2
    cos, sin = torch.cos(ang) / scl, torch.sin(ang) / scl                  # theta: salida -> entrada (escala > 1 = zoom in)
    theta = torch.stack([torch.stack([cos, -sin, tr[:, 0]], 1), torch.stack([sin, cos, tr[:, 1]], 1)], 1)  # (B,2,3)
    theta = theta.repeat_interleave(K_, 0)
    flat = x.reshape(B * K_, C, H, W)
    grid = F.affine_grid(theta, flat.shape, align_corners=False)
    return F.grid_sample(flat, grid, mode="bilinear", padding_mode="zeros", align_corners=False).view(B, K_, C, H, W)


def to_input(x_u8, gain, sag=None, train=False):
    x = x_u8.to(DEVICE, non_blocking=True).float() / 255.0
    if train:
        x = spatial_aug(x, sag)                                            # exp045: solo entrenamiento
    if not torch.all(gain == 1.0):
        x = (x * gain.to(DEVICE).view(-1, 1, 1, 1, 1)).clamp(0, 1)
    if NORM == "imagenet":
        x = (x - _MEAN.to(DEVICE)) / _STD.to(DEVICE)
    return x


def build_backbone(arch, pretrained=True):
    return timm.create_model(arch, pretrained=pretrained, num_classes=0, in_chans=3, global_pool="avg", img_size=RES)


class RaptorClassifier(nn.Module):
    """Raptor encoder + per-diagnosis attention-MIL head (12 findings)."""
    def __init__(self, backbone, F_dim=384, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K_ = x.shape[:2]
        return self.backbone(x.flatten(0, 1)).view(B, K_, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb

    def forward(self, x):
        return self.head(self.encode(x))


def build_model():
    bb = build_backbone(ARCH, pretrained=True)
    if GRAD_CKPT:
        bb.set_grad_checkpointing(True)
    return RaptorClassifier(bb, F_dim=bb.num_features).to(DEVICE)


# exp072: helpers puros, sampler con RNG privado y cabeza auxiliar separada.
FM_CELL = 32
assert RES % FM_CELL == 0 and FM_AUX_WINDOWS % (2 * FM_VOLS_PER_STEP) == 0


def fm_window_targets(boxes, crop, oy, ox, flip):
    """Targets de las cajas de tres cortes: cobertura por celda, presencia >=25 % y cajas conservadas."""
    grid = crop // FM_CELL
    dense = np.zeros((FM_A, grid, grid), np.float32)
    pres = np.zeros(FM_A, np.float32)
    if len(boxes) == 0:
        return dense, pres, np.zeros((0, 6), np.float64)
    b = np.asarray(boxes, np.float64)
    x0, x1 = np.clip(b[:, 1] - ox, 0, crop), np.clip(b[:, 3] - ox, 0, crop)
    y0, y1 = np.clip(b[:, 2] - oy, 0, crop), np.clip(b[:, 4] - oy, 0, crop)
    if flip:
        x0, x1 = crop - x1, crop - x0
    area = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    masks, kept = {}, []
    for i in np.flatnonzero(area > 0):
        cls = int(b[i, 0])
        mask = masks.setdefault(cls, np.zeros((crop, crop), np.float32))
        c0, c1 = int(math.floor(x0[i])), int(math.ceil(x1[i]))
        r0, r1 = int(math.floor(y0[i])), int(math.ceil(y1[i]))
        cc, rr = np.arange(c0, c1), np.arange(r0, r1)
        wx = np.minimum(x1[i], cc + 1) - np.maximum(x0[i], cc)
        wy = np.minimum(y1[i], rr + 1) - np.maximum(y0[i], rr)
        np.maximum(mask[r0:r1, c0:c1], np.outer(wy, wx).astype(np.float32), out=mask[r0:r1, c0:c1])
        if area[i] >= FM_PRESENCE_MIN * b[i, 5]:
            pres[cls] = 1.0
        kept.append((cls, x0[i], y0[i], x1[i], y1[i], area[i]))
    for cls, mask in masks.items():
        dense[cls] = mask.reshape(grid, FM_CELL, grid, FM_CELL).mean((1, 3))
    return dense, pres, np.asarray(kept, np.float64).reshape(-1, 6)


def fm_boxes_for_window(name, center):
    got = [FM_BOXES[(name, sl)] for sl in (center - 1, center, center + 1) if (name, sl) in FM_BOXES]
    return np.concatenate(got) if got else np.zeros((0, 6), np.float64)


class FmAuxBatches(torch.utils.data.IterableDataset):
    """Lotes auxiliares infinitos; cada worker usa solo su Generator de numpy, derivado de semilla y época."""
    def __init__(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        worker = torch.utils.data.get_worker_info()
        worker_id = 0 if worker is None else worker.id
        rng = np.random.default_rng(9800 + 100_000 * int(SEED) + 1_000 * self.epoch + worker_id)
        per_vol = FM_AUX_WINDOWS // FM_VOLS_PER_STEP
        while True:
            images = np.empty((FM_AUX_WINDOWS, 3, RES, RES), np.uint8)
            dense = np.empty((FM_AUX_WINDOWS, FM_A, RES // FM_CELL, RES // FM_CELL), np.float32)
            pres = np.empty((FM_AUX_WINDOWS, FM_A), np.float32)
            gains = rng.uniform(0.9, 1.1, size=FM_AUX_WINDOWS).astype(np.float32)
            out_i = 0
            for _ in range(FM_VOLS_PER_STEP):
                draw_boxed = rng.random() < FM_BOXED_FRAC
                pool = FM_TRAIN_BOXED if draw_boxed else FM_TRAIN_NEG
                name = pool[int(rng.integers(len(pool)))]
                n_slices = FM_N_SLICES[name]
                valid = np.arange(1, n_slices - 1, dtype=np.int64)
                if draw_boxed:
                    pos = FM_BOXED_SLICES[name]
                    pos = pos[(pos >= 1) & (pos < n_slices - 1)]
                    neg = np.setdiff1d(valid, pos, assume_unique=True)
                    pos = pos if len(pos) else valid
                    neg = neg if len(neg) else valid
                    n_pos = per_vol // 2
                    centers = np.concatenate([rng.choice(pos, n_pos), rng.choice(neg, per_vol - n_pos)])
                else:
                    centers = rng.choice(valid, per_vol)
                vol = np.load(Path(FM_LOCAL) / f"{name}.npy", mmap_mode="r")
                for center in centers:
                    center = int(center)
                    oy = int(rng.integers(0, IMG - RES + 1)); ox = int(rng.integers(0, IMG - RES + 1))
                    flip = bool(rng.random() < 0.5)
                    tri = np.asarray(vol[center - 1:center + 2, oy:oy + RES, ox:ox + RES])
                    if flip:
                        tri = tri[:, :, ::-1]
                    target_d, target_p, _ = fm_window_targets(fm_boxes_for_window(name, center), RES, oy, ox, flip)
                    images[out_i] = np.ascontiguousarray(tri)
                    dense[out_i], pres[out_i] = target_d, target_p
                    out_i += 1
                del vol
            yield (torch.from_numpy(images), torch.from_numpy(dense), torch.from_numpy(pres), torch.from_numpy(gains))


def fm_aux_loader(epoch):
    kwargs = {"dataset": FmAuxBatches(epoch), "batch_size": None, "num_workers": FM_WORKERS,
              "pin_memory": True, "persistent_workers": False,
              "generator": torch.Generator().manual_seed(9800 + int(SEED) + 1_000 * int(epoch))}
    if FM_WORKERS > 0:
        kwargs["prefetch_factor"] = 2
    return DataLoader(**kwargs)


class FmHoldoutWindows(Dataset):
    def __init__(self):
        self.specs = [(name, center) for name in FM_HOLDOUT for center in range(1, FM_N_SLICES[name] - 1)]
        self.current_name, self.current_vol = None, None

    def __len__(self):
        return len(self.specs)

    def __getitem__(self, index):
        name, center = self.specs[index]
        if name != self.current_name:
            self.current_name = name
            self.current_vol = np.load(Path(FM_LOCAL) / f"{name}.npy", mmap_mode="r")
        offset = (IMG - RES) // 2
        tri = np.asarray(self.current_vol[center - 1:center + 2, offset:offset + RES, offset:offset + RES])
        _, pres, _ = fm_window_targets(fm_boxes_for_window(name, center), RES, offset, offset, False)
        return torch.from_numpy(np.ascontiguousarray(tri)), torch.from_numpy(pres)


class FmAuxHead(nn.Module):
    """LayerNorm(d) -> Linear(d, 11) por posición; parámetros fuera de RaptorClassifier."""
    def __init__(self, d):
        super().__init__()
        self.norm = nn.LayerNorm(d, eps=1e-6)
        generator = torch.Generator().manual_seed(9800)
        self.weight = nn.Parameter(torch.randn(FM_A, d, generator=generator) * 0.02)
        self.bias = nn.Parameter(torch.full((FM_A,), -4.0))

    def forward(self, fmap):
        assert fmap.ndim == 4
        with torch.autocast(device_type=fmap.device.type, enabled=False):
            features = self.norm(fmap.permute(0, 2, 3, 1).float())
            logits = F.linear(features, self.weight, self.bias)
        return logits.permute(0, 3, 1, 2)


def fm_backbone_map(model, x, freeze_bn_running=False):
    """Forward espacial NCHW. momentum=0 evita que fastMRI mueva running stats, pero conserva batch stats de train."""
    bns = [m for m in model.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)] if freeze_bn_running else []
    momenta = [m.momentum for m in bns]
    for module in bns:
        module.momentum = 0.0
    try:
        fmap = model.backbone.forward_features(x)
    finally:
        for module, momentum in zip(bns, momenta):
            module.momentum = momentum
    assert fmap.ndim == 4, f"fastMRI+ necesita mapa 4D, recibido {tuple(fmap.shape)}"
    d = model.backbone.num_features
    if fmap.shape[1] == d:
        pass
    elif fmap.shape[-1] == d:
        fmap = fmap.permute(0, 3, 1, 2)
    else:
        raise AssertionError(f"canal del mapa {tuple(fmap.shape)} no coincide con num_features={d}")
    grid = RES // FM_CELL
    assert tuple(fmap.shape[-2:]) == (grid, grid), (tuple(fmap.shape), grid)
    return fmap


def fm_presence_logits(dense_logits):
    return torch.logsumexp(dense_logits.flatten(2), -1) - math.log(dense_logits.shape[-2] * dense_logits.shape[-1])


## 4. Entrenamiento (bucle del script público; selección y top-K por pseudo L2 en val, nunca gold)

In [ ]:
from sklearn.metrics import roc_auc_score


def loader(ids, k, train, bs):
    return DataLoader(StudyWindows(ids, k, train), batch_size=bs, shuffle=train, drop_last=train,
                      num_workers=N_WORKERS, pin_memory=True, persistent_workers=False)


@torch.no_grad()
def predict(model, ids):
    model.eval()
    if not ids:
        return pd.DataFrame(columns=LABELS)
    out = np.zeros((len(ids), len(LABELS)), np.float32)
    for x, g, idx, _sag in loader(ids, K_EVAL, False, max(2, BATCH_SIZE // 2)):   # exp045: sin aug
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            out[idx.numpy()] = torch.sigmoid(model(to_input(x, g)).float()).cpu().numpy()
    return pd.DataFrame(out, index=pd.Index(ids, name="StudyInstanceUID"), columns=LABELS)


def score(pred):
    r = {}
    for name, ref in (("pseudo", pseudo_bin), ("v1", v1_bin), ("gold", gold)):
        ids = [u for u in pred.index if u in ref.index]
        r[name] = mean_auc(ref.loc[ids, LABELS].to_numpy(), pred.loc[ids, LABELS].to_numpy()) if ids else None
    return r


def average_states(states):
    avg = {}
    for k_ in states[0]:
        v0 = states[0][k_]
        avg[k_] = (sum(s[k_].double() for s in states) / len(states)).to(v0.dtype) if v0.is_floating_point() else v0.clone()
    return avg


def save_preds(pred_val, pred_gold, name):
    if len(pred_val):
        pred_val.add_prefix("pred_").reset_index().to_csv(RESULTS / f"oof_{name}.csv", index=False)
    pred_gold.add_prefix("pred_").reset_index().to_csv(RESULTS / f"gold_{name}.csv", index=False)


@torch.no_grad()
def fm_holdout_presence_auc(model, fm_head):
    """AUC por clase a nivel ventana, con todos los centros válidos, crop central y sin augmentation."""
    was_model, was_head = model.training, fm_head.training
    model.eval(); fm_head.eval()
    ds = FmHoldoutWindows()
    kwargs = {"dataset": ds, "batch_size": FM_AUX_WINDOWS, "shuffle": False, "num_workers": FM_WORKERS,
              "pin_memory": True, "persistent_workers": False,
              "generator": torch.Generator().manual_seed(9800)}
    if FM_WORKERS > 0:
        kwargs["prefetch_factor"] = 2
    pred, target = [], []
    for x, y in DataLoader(**kwargs):
        gain = torch.ones(len(x), dtype=torch.float32)
        x = to_input(x.unsqueeze(1), gain, train=False).squeeze(1)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            fmap = fm_backbone_map(model, x)
        pred.append(torch.sigmoid(fm_presence_logits(fm_head(fmap))).cpu())
        target.append(y)
    model.train(was_model); fm_head.train(was_head)
    pred = torch.cat(pred).numpy(); target = torch.cat(target).numpy()
    auc = {}
    for i, name in enumerate(FM_CLASSES):
        y = target[:, i]
        auc[name] = float(roc_auc_score(y, pred[:, i])) if y.max() > 0 and y.min() < 1 else None
    print("fastMRI+ holdout AUC (ventana):", auc, flush=True)
    return auc


def train_fold(fold):
    seed_everything(SEED + fold)
    tr_ids, va_ids = fold_ids(fold)
    if SMOKE:
        tr_ids, va_ids = tr_ids[:64], va_ids[:48]
    sfx = f"{TAG}{'' if ALL_DATA else fold}"
    tl = loader(tr_ids, K, True, BATCH_SIZE)
    model = build_model()
    fm_head = FmAuxHead(model.backbone.num_features).to(DEVICE) if FM_AUX_WEIGHT > 0 else None
    fm_pos_weight = torch.tensor(FM_POS_WEIGHT, device=DEVICE) if fm_head is not None else None
    prev = np.clip(np.nanmean(soft.loc[tr_ids, LABELS].to_numpy(), 0), 0.03, 0.7)
    lossf = nn.BCEWithLogitsLoss(reduction="none", pos_weight=torch.tensor(np.clip((1 - prev) / prev, 1, 10), dtype=torch.float32, device=DEVICE))
    Y = {u: torch.tensor(soft.loc[u, LABELS].to_numpy(), dtype=torch.float32) for u in tr_ids}
    head_p = [p for n_, p in model.named_parameters() if not n_.startswith("backbone.")]
    bb_p = [p for n_, p in model.named_parameters() if n_.startswith("backbone.")]
    groups = [{"params": bb_p, "lr": BB_LR}, {"params": head_p, "lr": HEAD_LR}]
    max_lr = [BB_LR, HEAD_LR]
    if fm_head is not None:
        groups.append({"params": list(fm_head.parameters()), "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY})
        max_lr.append(HEAD_LR)
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, total_steps=max(1, len(tl) * EPOCHS), pct_start=PCT_START)
    print(f"[{sfx}] train {len(tr_ids)} · val {len(va_ids)} · {len(tl)} pasos/época · pos_weight "
          f"{np.round(np.clip((1 - prev) / prev, 1, 10), 2).tolist()}", flush=True)
    if fm_head is not None:
        print(f"[{sfx}] fastMRI+ {FM_AUX_WINDOWS} ventanas/paso · {FM_VOLS_PER_STEP} volúmenes · "
              f"cabeza {sum(p.numel() for p in fm_head.parameters())} parámetros", flush=True)
    history, topk, best, keep = [], [], None, {}
    fm_dense_sum = fm_pres_sum = fm_ms_total = 0.0
    fm_steps_total, fm_pending = 0, []

    def flush_fm_timing():
        nonlocal fm_ms_total, fm_pending
        if fm_pending:
            fm_pending[-1][1].synchronize()
            fm_ms_total += sum(start_ev.elapsed_time(end_ev) + fetch_ms for start_ev, end_ev, fetch_ms in fm_pending)
            fm_pending = []

    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        if fm_head is not None:
            fm_head.train()
            fm_dl = fm_aux_loader(epoch)
            fm_it = iter(fm_dl)
        tot = 0.0; t_ep = time.time()
        for step, (x, g, idx, sag) in enumerate(tl):
            y = torch.stack([Y[tl.dataset.ids[i]] for i in idx.tolist()]).to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
                logits = model(to_input(x, g, sag, train=True))   # exp045: aug espacial solo en train
            m = ~torch.isnan(y)                       # NaN enmascarado (= media BCE del padre si no hay NaN)
            loss = (lossf(logits.float(), torch.nan_to_num(y)) * m).sum() / m.sum().clamp(min=1)
            loss.backward()

            if fm_head is not None:
                fetch_t0 = time.perf_counter()
                x_fm, y_dense, y_pres, fm_gain = next(fm_it)
                fetch_ms = (time.perf_counter() - fetch_t0) * 1000.0
                if DEVICE.type == "cuda":
                    start_ev, end_ev = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
                    start_ev.record()
                else:
                    aux_t0 = time.perf_counter()
                devices = [torch.cuda.current_device()] if DEVICE.type == "cuda" else []
                with torch.random.fork_rng(devices=devices):
                    aux_seed = 9800 + 1_000_000 * int(fold) + 10_000 * int(epoch) + int(step)
                    torch.random.default_generator.manual_seed(aux_seed)
                    if DEVICE.type == "cuda":
                        torch.cuda.manual_seed(aux_seed)
                    x_fm = to_input(x_fm.unsqueeze(1), fm_gain, train=False).squeeze(1)
                    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
                        fmap = fm_backbone_map(model, x_fm, freeze_bn_running=True)
                    dense_logits = fm_head(fmap)
                    y_dense = y_dense.to(DEVICE, non_blocking=True)
                    y_pres = y_pres.to(DEVICE, non_blocking=True)
                    loss_dense = F.binary_cross_entropy_with_logits(
                        dense_logits, y_dense, pos_weight=fm_pos_weight)
                    presence_logits = fm_presence_logits(dense_logits)
                    loss_presence = F.binary_cross_entropy_with_logits(presence_logits, y_pres)
                    (FM_AUX_WEIGHT * (loss_dense + loss_presence)).backward()
                if DEVICE.type == "cuda":
                    end_ev.record(); fm_pending.append((start_ev, end_ev, fetch_ms))
                else:
                    fm_ms_total += (time.perf_counter() - aux_t0) * 1000.0 + fetch_ms
                fm_dense_sum += float(loss_dense.detach()); fm_pres_sum += float(loss_presence.detach())
                fm_steps_total += 1

            if fm_head is None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            else:
                torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(fm_head.parameters()), GRAD_CLIP)
            opt.step(); sched.step(); tot += loss.item()
            if (step + 1) % LOG_EVERY == 0:
                el = time.time() - t_ep
                print(f"  ep{epoch} step {step + 1}/{len(tl)} loss {tot / (step + 1):.4f} · {el / (step + 1):.2f} s/paso · "
                      f"ETA época {(len(tl) - step - 1) * el / (step + 1) / 60:.1f} min", flush=True)
                if fm_head is not None:
                    flush_fm_timing()
                    print(f"    fastMRI+ L_aux {fm_dense_sum / fm_steps_total + fm_pres_sum / fm_steps_total:.4f} "
                          f"(dense {fm_dense_sum / fm_steps_total:.4f}, presencia {fm_pres_sum / fm_steps_total:.4f}) · "
                          f"{fm_ms_total / fm_steps_total:.1f} ms/paso", flush=True)
        if fm_head is not None:
            flush_fm_timing()
            del fm_it, fm_dl
        t_train = time.time() - t_ep
        pv, pg = predict(model, va_ids), predict(model, gold_ids)
        sv, sg = score(pv), score(pg)
        rec = {"epoch": epoch, "train_loss": tot / max(1, len(tl)),
               "pseudo_auc": sv["pseudo"]["mean"] if sv["pseudo"] else None,
               "v1_auc": sv["v1"]["mean"] if sv["v1"] else None,
               "gold_auc": sg["gold"]["mean"], "train_s": t_train, "eval_s": time.time() - t_ep - t_train,
               "minutes": (time.time() - start) / 60}
        if fm_head is not None:
            rec.update({"fm_aux_dense": fm_dense_sum / fm_steps_total, "fm_aux_presence": fm_pres_sum / fm_steps_total,
                        "fm_aux_ms_per_step": fm_ms_total / fm_steps_total})
        history.append(rec)
        pd.DataFrame(history).to_csv(RESULTS / f"history_{sfx}.csv", index=False)
        print(f"[{sfx}] ep{epoch} loss {rec['train_loss']:.4f} | pseudo L2 {rec['pseudo_auc'] or float('nan'):.4f} | "
              f"v1 {rec['v1_auc'] or float('nan'):.4f} | gold {rec['gold_auc']:.4f} (solo veto) | train {t_train:.0f}s "
              f"eval {rec['eval_s']:.0f}s | {rec['minutes']:.1f} min", flush=True)
        state = {k_: v.detach().cpu().clone() for k_, v in model.state_dict().items()}
        if ALL_DATA:
            if epoch in SWA_EPOCHS:
                keep[epoch] = state
            torch.save({"model": state, "arch": ARCH, "res": RES, "lab": LABELS, "epoch": epoch, "exp": EXP},
                       RESULTS / f"last_{sfx}.pt")
            continue
        crit = rec["pseudo_auc"]
        if best is None or crit > best["pseudo_auc"]:
            best = dict(rec)
            torch.save({"model": state, "arch": ARCH, "res": RES, "lab": LABELS, "epoch": epoch, "exp": EXP,
                        "pseudo_auc": crit, "params": PARAMS}, RESULTS / f"best_{sfx}.pt")
            save_preds(pv, pg, f"best_{sfx}")
            best["pseudo_per_label"], best["gold_per_label"] = sv["pseudo"]["per_label"], sg["gold"]["per_label"]
        topk.append((crit, epoch, state)); topk.sort(key=lambda t: -t[0]); del topk[TOPK:]
        del state
    gpu_h = (time.time() - start) / 3600

    fm_auc = None
    if fm_head is not None:
        # Diagnóstico con los pesos de la última época, antes de cargar SWA; nunca participa en selección.
        fm_auc = fm_holdout_presence_auc(model, fm_head)
        head_name = "fm_aux_head_alldata.pt" if ALL_DATA else f"fm_aux_head_f{fold}.pt"
        torch.save({"model": {k_: v.detach().cpu() for k_, v in fm_head.state_dict().items()},
                    "classes": FM_CLASSES, "epoch": EPOCHS - 1, "exp": EXP}, RESULTS / head_name)

    if ALL_DATA:
        eps = sorted(keep) or [EPOCHS - 1]
        states = [keep[e] for e in sorted(keep)] or [model.state_dict()]
    else:
        eps = [e for _, e, _ in topk]
        states = [s for _, _, s in topk]
    avg = average_states(states)
    model.load_state_dict(avg)
    pv, pg = predict(model, va_ids), predict(model, gold_ids)
    sv, sg = score(pv), score(pg)
    torch.save({"model": avg, "arch": ARCH, "res": RES, "lab": LABELS, "swa_over": eps, "exp": EXP, "params": PARAMS},
               RESULTS / f"swa_{sfx}.pt")
    save_preds(pv, pg, f"swa_{sfx}")
    swa = {"epochs": eps, "pseudo_auc": sv["pseudo"]["mean"] if sv["pseudo"] else None,
           "v1_auc": sv["v1"]["mean"] if sv["v1"] else None, "gold_auc": sg["gold"]["mean"],
           "pseudo_per_label": sv["pseudo"]["per_label"] if sv["pseudo"] else None, "gold_per_label": sg["gold"]["per_label"]}
    print(f"[{sfx}] SWA sobre épocas {eps} | pseudo L2 {swa['pseudo_auc'] or float('nan'):.4f} | "
          f"v1 {swa['v1_auc'] or float('nan'):.4f} | gold {swa['gold_auc']:.4f}", flush=True)
    out = {"n_train": len(tr_ids), "n_val": len(va_ids), "history": history, "swa": swa, "gpu_hours": gpu_h,
           "swa_ckpt_path": str(RESULTS / f"swa_{sfx}.pt"), "fm_aux_holdout_auc": fm_auc,
           "fm_aux_ms_per_step": fm_ms_total / fm_steps_total if fm_steps_total else None}
    if best is not None:
        out.update({"pseudo_auc": best["pseudo_auc"], "v1_auc": best["v1_auc"], "gold_auc": best["gold_auc"],
                    "best_epoch": best["epoch"], "pseudo_per_label": best["pseudo_per_label"],
                    "gold_per_label": best["gold_per_label"], "ckpt_path": str(RESULTS / f"best_{sfx}.pt")})
    return out


PARAMS = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, bool))}
per_fold = {}
for fold in FOLDS:
    per_fold["alldata" if ALL_DATA else str(fold)] = train_fold(fold)


## 5. Evaluación (pseudo L2 decide; v1 y gold informativos/veto)

In [ ]:
rows = {}
for f, r in per_fold.items():
    rows[f"{f} mejor época ({r.get('best_epoch')})"] = {"pseudo L2": r.get("pseudo_auc"), "v1": r.get("v1_auc"), "gold (veto)": r.get("gold_auc")}
    rows[f"{f} SWA {r['swa']['epochs']}"] = {"pseudo L2": r["swa"]["pseudo_auc"], "v1": r["swa"]["v1_auc"], "gold (veto)": r["swa"]["gold_auc"]}
summary = pd.DataFrame(rows).T
display(summary.round(4))
parent = (leer_metricas(PADRE) if PADRE else None) or {}
r0 = per_fold.get("0", {})
parent_f0 = parent.get("per_fold", {}).get("0", {}).get("pseudo_auc")
delta_pseudo = r0.get("pseudo_auc") - parent_f0 if parent_f0 is not None and r0.get("pseudo_auc") is not None else None
parent_gold = parent.get("per_fold", {}).get("0", {}).get("gold_auc")
delta_gold = r0.get("gold_auc") - parent_gold if parent_gold is not None and r0.get("gold_auc") is not None else None
DECISION = decision(delta_pseudo, delta_gold)
print("Decisión automática:", DECISION, "(sin padre: es una reproducción; ver criterios absolutos en SPEC)")
for f, r in per_fold.items():
    display(pd.DataFrame(r["history"]).set_index("epoch").round(4))

## 6. Guardar `results/<EXP>/metrics.json`

In [ ]:
metrics = {
    "schema_version": 1, "exp": EXP, "author": AUTHOR, "parent": PADRE,
    "split": SPLIT, "split_sha256": split.attrs["sha256"], "git": GIT,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "smoke" if SMOKE else ("alldata" if ALL_DATA else "screen"),
    "select_on": "pseudo_L2_bin>0.5", "params": PARAMS, "per_fold": per_fold,
    "gold_auc": r0.get("gold_auc"), "gold_per_label": r0.get("gold_per_label"), "oof_pseudo_auc": None,
    "swa_f0": r0.get("swa"),
    "delta": {"pseudo_f0": delta_pseudo, "gold": delta_gold},
    "decision_auto": DECISION,
    "corpus": {"path": CORPUS, "n": len(corpus_ids), "D": D, "img": IMG, "build_seconds": meta.get("seconds"),
               "n_errors": len(meta["errors"])},
    "fm_aux": {"weight": FM_AUX_WEIGHT, "windows": FM_AUX_WINDOWS, "vols_per_step": FM_VOLS_PER_STEP,
               "n_train_vols": len(FM_TRAIN_BOXED) + len(FM_TRAIN_NEG), "n_holdout_vols": len(FM_HOLDOUT),
               "boxes_per_class": FM_BOX_COUNTS,
               "ms_per_step": {f: r.get("fm_aux_ms_per_step") for f, r in per_fold.items()},
               "holdout_auc": {f: r.get("fm_aux_holdout_auc") for f, r in per_fold.items()}},
    "gpu_hours_total": sum(f["gpu_hours"] for f in per_fold.values()),
}
# all-data no pisa el metrics.json del fold 0: va a results/<EXP>/alldata/metrics.json
print("Guardado:", guardar_metricas(RESULTS / "alldata" if ALL_DATA else RESULTS, metrics))


## 7. Verdict

Copia los números de la celda 5 a la sección 3 de `SPEC.md` y actualiza la fila en `docs/EXPERIMENTOS.md`
(estado `worked`, `didnt` o `killed`). Después: commit y PR a `main`.